##Import Libraries

In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import shapiro
import math
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from scipy import signal
from sklearn.model_selection import GridSearchCV
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.cross_decomposition import PLSRegression
from xgboost.sklearn import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor
from catboost import CatBoostRegressor
from sklearn.feature_selection import SelectKBest, mutual_info_regression, f_regression
from sklearn.model_selection import cross_val_score
from scipy.stats import pearsonr
import itertools
from sklearn.feature_selection import RFE
from sklearn.feature_selection import RFECV
from numpy import random

In [3]:
import ee
import requests
import numpy
import numpy as np
import csv
from PIL import Image
import geemap
import matplotlib
import matplotlib.pyplot as plt
import datetime
import math
import pandas as pd
from io import BytesIO
from google.cloud import storage
import os
import itertools
from flask import jsonify
import json

## Inicialize Google Earth Engine Credentials

In [ ]:
try:
       ee.Initialize()
except:
       service_account = '' #Insert your service account
       credentials = ee.ServiceAccountCredentials(service_account, '')#Insert here the autentication json
       ee.Initialize(credentials)

## Calculate new dates

In [ ]:
def cal_day_of_year(current_date):
    current_date2 = datetime.datetime.strptime(current_date, "%Y-%m-%d")
    day_of_year = current_date2.timetuple().tm_yday
    return day_of_year

## Check Vegetation Indeces Values

In [ ]:
def check_if_some_index_is_nan(NDVI, NDWI, EVI, LAI, DVI, GCI, GEMI, SAVI):
    list_of_bands = [NDVI, NDWI, EVI, LAI, DVI, GCI, GEMI, SAVI]
    for current_band in list_of_bands:
        if math.isnan(current_band) == 1:
            return 0
    return 1

## Calculate Vegetation Indeces Values

In [ ]:
def calc_ndvi(NIR, RED):
    return (NIR-RED)/(NIR+RED)

def calc_ndwi(RED_EDGE, SWIR1):
    return (RED_EDGE-SWIR1)/(RED_EDGE+SWIR1)

def calc_evi_LAI(NIR, RED):
    EVI = 2.5*(NIR-RED)/(NIR + 2.4*RED + 10000)
    LAI = (3.618*EVI-0.118)
    return EVI, LAI

def calc_GEMI(NIR, RED):
    eta = (2 * (NIR * NIR - RED * RED) + 1.5 * NIR + 0.5 * RED) / (RED + NIR + 0.5)
    return eta*(1-0.25*eta) - ((RED - 0.125)/(1 - RED))

def calc_SAVI(RED, NIR):
    return 1.5*(NIR-RED)/(RED+NIR+0.5)

def calc_DVI(RED, NIR):
    return NIR - RED

def calc_GCI(NIR, GREEN):
    return ((NIR/GREEN)-1)

## Calculate Average of Bands Reflectance

In [ ]:
def convert_bands_in_colors_average(B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B11, B12):

    B1 = sum(B1) / len(B1)
    B2 = sum(B2) / len(B2)
    B3 = sum(B3) / len(B3)
    B4 = sum(B4) / len(B4)
    B5 = sum(B5) / len(B5)
    B6 = sum(B6) / len(B6)
    B7 = sum(B7) / len(B7)
    B8 = sum(B8) / len(B8)
    B8A = sum(B8A) / len(B8A)
    B9 = sum(B9) / len(B9)
    B11 = sum(B11) / len(B11)
    B12 = sum(B12) / len(B12)


    return B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B11, B12

## Calculate Bands Values

In [ ]:
def LatLonImg_shape(img, local_estudo):
    img = img.clip(local_estudo).mask(mask=1)
    img = img.addBands(ee.Image.pixelLonLat())
    img = img.addBands(ee.Image.pixelArea())
    img = img.reduceRegion(reducer=ee.Reducer.toList(), \
                           geometry=local_estudo, \
                           maxPixels=1e13, \
                           crs= 'EPSG:4674',\
                           scale=10);

    area = np.array((ee.Array(img.get("area")).getInfo()))
    lats = np.array((ee.Array(img.get("latitude")).getInfo()))
    lons = np.array((ee.Array(img.get("longitude")).getInfo()))
    B1 = np.array((ee.Array(img.get("B1")).getInfo()))
    B2 = np.array((ee.Array(img.get("B2")).getInfo()))
    B3 = np.array((ee.Array(img.get("B3")).getInfo()))
    B4 = np.array((ee.Array(img.get("B4")).getInfo()))
    B5 = np.array((ee.Array(img.get("B5")).getInfo()))
    B6 = np.array((ee.Array(img.get("B6")).getInfo()))
    B7 = np.array((ee.Array(img.get("B7")).getInfo()))
    B8 = np.array((ee.Array(img.get("B8")).getInfo()))
    B8A = np.array((ee.Array(img.get("B8A")).getInfo()))
    B9 = np.array((ee.Array(img.get("B9")).getInfo()))
    B11 = np.array((ee.Array(img.get("B11")).getInfo()))
    B12 = np.array((ee.Array(img.get("B12")).getInfo()))

    return lats, lons, img, B1.astype(float), B2.astype(float), \
           B3.astype(float), B4.astype(float), B5.astype(float), B6.astype(float), \
           B7.astype(float), B8.astype(float), B8A.astype(float), B9.astype(float), \
           B11.astype(float), B12.astype(float), area

## Reproject the image to EPSG:4674

In [ ]:
def reprojetation(img):
  proj = ee.Projection('EPSG:4674');
  imgr = img.reproject(proj, None, 10)
  return img

## Calculate the Previous Data

In [ ]:
def calc_previous_date(curr_date):
    curr_date_temp = datetime.datetime.strptime(curr_date, "%Y-%m-%d")
    next_date = curr_date_temp - datetime.timedelta(days=1)
    next_date = next_date.strftime("%Y-%m-%d")
    return next_date

## Calculate the Image and Dates

In [ ]:
def returnImagesAndDatePixel(date, geom):

    flag_find_image = 0
    while(flag_find_image!=1):
        InitDate = calc_previous_date(date)
        col = ee.ImageCollection("COPERNICUS/S2_SR") \
            .filterDate(InitDate, date) \
            .filterBounds(geom) \
            .select(['B1', 'B5', 'B6', 'B7', 'B8', 'B4', 'B3', 'B2', 'B8A', 'B9', 'B11', 'B12']) \
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 50))

        dates = ee.List(col \
                        .aggregate_array('system:time_start')) \
            .map(lambda time_start:
                 ee.Date(time_start).format('Y-MM-dd')
                 ) \
            .getInfo()
        if len(dates) > 0:
            flag_find_image = 1
        else:
            date = InitDate
    return col, dates

## Create the Dataset

In [ ]:
def get_X_data(alldates, shapefile, period):
    general_images_date_caracteristics = []
    image_atual = 0
    vector_images_with_problems = []
    flag_calc_area = True
    dates_american_format = []
    for date in alldates:
        dates_american_format.append(date)
        col, dates = returnImagesAndDatePixel(date, shapefile)
        ListImages = col.toList(col.size())
        ListSize = ListImages.size().getInfo()

        img0 = ee.Image(ListImages.get(0))
        img0 = reprojetation(img0)

        lats, lons, img, B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B11, B12, pixel_area = LatLonImg_shape(img0, shapefile)


        B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B11, B12 = convert_bands_in_colors_average(B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B11, B12)

        if flag_calc_area == True:
            paddock_area = str(sum(pixel_area)*0.0001)
            flag_calc_area = False


        NIR = B8.astype(float)
        RED = B4.astype(float)
        RED_EDGE = B8A.astype(float)
        SWIR1 = B11.astype(float)
        GREEN = B3.astype(float)

        NDVI = calc_ndvi(NIR, RED)
        NDWI = calc_ndwi(RED_EDGE, SWIR1)
        EVI, LAI = calc_evi_LAI(NIR, RED)
        DVI = calc_DVI(RED, NIR)
        GCI = calc_GCI(NIR, GREEN)
        GEMI = calc_GEMI(NIR, RED)
        SAVI = calc_SAVI(RED, NIR)

        if check_if_some_index_is_nan(NDVI, NDWI, EVI, LAI, DVI, GCI, GEMI, SAVI) == 1:
            image_date_caracteristics = []
            real_date = cal_day_of_year(date)
            image_date_caracteristics.append(period)
            image_date_caracteristics.append(real_date)
            image_date_caracteristics.append(B1)
            image_date_caracteristics.append(B2)
            image_date_caracteristics.append(B3)
            image_date_caracteristics.append(B4)
            image_date_caracteristics.append(B5)
            image_date_caracteristics.append(B6)
            image_date_caracteristics.append(B7)
            image_date_caracteristics.append(B8)
            image_date_caracteristics.append(B8A)
            image_date_caracteristics.append(B9)
            image_date_caracteristics.append(B11)
            image_date_caracteristics.append(B12)
            image_date_caracteristics.append(NDVI)
            image_date_caracteristics.append(NDWI)
            image_date_caracteristics.append(EVI)
            image_date_caracteristics.append(LAI)
            image_date_caracteristics.append(DVI)
            image_date_caracteristics.append(GCI)
            image_date_caracteristics.append(GEMI)
            image_date_caracteristics.append(SAVI)
            image_date_caracteristics.append(0)
            general_images_date_caracteristics.append(image_date_caracteristics)
        else:
            vector_images_with_problems.append(image_atual)
        image_atual = image_atual + 1
    new_df = pd.DataFrame(columns=['period','real_date', 'B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7',
                                       'B8', 'B8A', 'B9', 'B11', 'B12', 'NDVI', 'NDWI', 'EVI',
                                       'LAI', 'DVI', 'GCI', 'GEMI', 'SAVI', 'Animals'])
    for i in range(0, len(general_images_date_caracteristics)):
        new_df.loc[len(new_df)] = general_images_date_caracteristics[i]
    return new_df, vector_images_with_problems, paddock_area, dates_american_format

## Load .shp (shapefile)

In [ ]:
##Insert here shapefile path
shapefile = geemap.shp_to_ee("D:/Experimento - Efeeder/Shapefiles-20231130T142130Z-001/Shapefiles/All-Pontos/All-Poligono.shp")

## Calculate the list of dates for every experimentation period

In [ ]:
def generate_list_of_dates(init, end, step):
    curr_date_temp = datetime.datetime.strptime(init, "%d/%m/%Y")
    dates = []
    end_date =  datetime.datetime.strptime(end, "%d/%m/%Y")
    while curr_date_temp <= end_date:
        date = str(curr_date_temp).split(" ")
        date = date[0]
        dates.append(str(date))
        next_date = curr_date_temp + datetime.timedelta(days=step)
        curr_date_temp = next_date.strftime("%d/%m/%Y")
        curr_date_temp = datetime.datetime.strptime(curr_date_temp, "%d/%m/%Y")
    return dates

## Experimentation Periods

In [ ]:
dicio_period = {"period 1": ["23/12/2022", "20/01/2023"],
                "period 2": ["21/01/2023", "16/02/2023"],
                "period 3": ["17/02/2023", "07/03/2023"],
                "period 4": ["08/03/2023", "21/03/2023"],
                "period 5": ["22/03/2023", "12/04/2023"],
                "period 6": ["13/04/2023", "28/04/2023"],
                "period 7": ["29/04/2023", "11/05/2023"],
                "period 8": ["12/05/2023", "30/05/2023"],
                "period 9": ["31/05/2023", "16/06/2023"],
                "period 10": ["17/06/2023", "11/07/2023"],
                "period 11": ["12/07/2023", "03/08/2023"],
                "period 12": ["04/08/2023", "22/08/2023"],
                "period 13": ["23/08/2023", "15/09/2023"],
                "period 14": ["16/09/2023", "04/10/2023"],
                "period 15": ["05/10/2023","26/10/2023"]}

## Generate all the data

In [ ]:
dicio_period_dataset = {"period 1": 0,
                "period 2": 0,
                "period 3": 0,
                "period 4": 0,
                "period 5": 0,
                "period 6": 0,
                "period 7": 0,
                "period 8": 0,
                "period 9": 0,
                "period 10": 0,
                "period 11": 0,
                "period 12": 0,
                "period 13": 0,
                "period 14": 0,
                "period 15": 0}

for period in dicio_period.items():
    dates = generate_list_of_dates(period[1][0], period[1][1], 5)
    X, vector_images_with_problems, paddock_area, dates_american_format = get_X_data(dates, shapefile, period[0])
    dicio_period_dataset[period[0]] = X


2022-12-23
2022-12-28
2023-01-02
2023-01-07
2023-01-12
2023-01-17
2023-01-21
2023-01-26
2023-01-31
2023-02-05
2023-02-10
2023-02-15
2023-02-17
2023-02-22
2023-02-27
2023-03-04
2023-03-08
2023-03-13
2023-03-18
2023-03-22
2023-03-27
2023-04-01
2023-04-06
2023-04-11
2023-04-13
2023-04-18
2023-04-23
2023-04-28
2023-04-29
2023-05-04
2023-05-09
2023-05-12
2023-05-17
2023-05-22
2023-05-27
2023-05-31
2023-06-05
2023-06-10
2023-06-15
2023-06-17
2023-06-22
2023-06-27
2023-07-02
2023-07-07
2023-07-12
2023-07-17
2023-07-22
2023-07-27
2023-08-01
2023-08-04
2023-08-09
2023-08-14
2023-08-19
2023-08-23
2023-08-28
2023-09-02
2023-09-07
2023-09-12
2023-09-16
2023-09-21
2023-09-26
2023-10-01
2023-10-05
2023-10-10
2023-10-15
2023-10-20
2023-10-25


## Create the Total Dataset

In [ ]:
current_Dataframe = dicio_period_dataset["period 1"]
for period in dicio_period_dataset.items():
    if period[0]!="period 1":
        current_Dataframe = pd.concat([current_Dataframe, period[1]])


<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pand

## Datset per period

In [23]:
current_Dataframe

,PERIOD,BAND1,BAND2,BAND3,BAND4,BAND5,BAND6,BAND7,BAND8,BAND8A,...,BAND11,BAND12,NDVI_INDEX,NDWI_INDEX,EVI_INDEX,LAI_INDEX,DVI_INDEX,GCI_INDEX,GEMI_INDEX,SAVI_INDEX
0,period 1,11132.281076,10321.069628,9713.271252,9424.406664,10205.371636,9788.023494,9603.466894,9377.225117,9432.536523,...,6943.019223,6089.964972,-0.002509,0.152026,-0.002809,-0.128162,-47.181546,-0.034597,-2.271471e+03,-0.003764
1,period 1,11132.281076,10321.069628,9713.271252,9424.406664,10205.371636,9788.023494,9603.466894,9377.225117,9432.536523,...,6943.019223,6089.964972,-0.002509,0.152026,-0.002809,-0.128162,-47.181546,-0.034597,-2.271471e+03,-0.003764
2,period 1,1517.104656,1630.292610,1986.546775,1943.295173,2620.463477,3645.244340,3924.217001,3793.000427,4009.959419,...,3360.655703,2591.492525,0.322456,0.088094,0.250544,0.788467,1849.705254,0.909344,-3.419260e+06,0.483642
3,period 1,1517.104656,1630.292610,1986.546775,1943.295173,2620.463477,3645.244340,3924.217001,3793.000427,4009.959419,...,3360.655703,2591.492525,0.322456,0.088094,0.250544,0.788467,1849.705254,0.909344,-3.419260e+06,0.483642
4,period 1,1401.452371,1512.057668,1906.866724,1816.693721,2587.626655,3679.461341,3911.602734,3801.595045,3987.941478,...,3371.013669,2551.551901,0.353293,0.083834,0.273227,0.870535,1984.901324,0.993634,-3.937496e+06,0.529892
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62,period 15,4924.431012,4403.952157,4211.138830,4123.635199,4561.456215,4683.868432,4739.846647,4600.068347,4862.510466,...,4805.788125,4272.423751,0.054614,0.005867,0.048622,0.057914,476.433148,0.092357,-2.264973e+05,0.081916
63,period 15,1492.388723,1628.452371,1892.912003,2129.825716,2576.218283,3123.737292,3378.120461,3322.640752,3592.108928,...,3849.788125,3000.378898,0.218766,-0.034625,0.161766,0.467271,1192.815036,0.755307,-1.421483e+06,0.328119
64,period 15,1508.269970,1564.428877,1871.449808,2069.979069,2599.031183,3151.041862,3344.433575,3296.873985,3554.879111,...,3865.520290,2978.706963,0.228606,-0.041863,0.167931,0.489576,1226.894917,0.761668,-1.503902e+06,0.342877
65,period 15,1508.292183,1628.410081,1883.325929,2207.646305,2645.384878,3107.100384,3340.459205,3270.243486,3539.678343,...,4027.446390,3223.490816,0.193979,-0.064459,0.143064,0.399605,1062.597181,0.736419,-1.127945e+06,0.290942


## Save the dataset

In [24]:
current_Dataframe.to_csv("Table_2_Field_Multispectral_Data.csv", sep=',', index=False, encoding='utf-8')

In [22]:
# save csv in the dataframe
current_Dataframe = pd.read_csv("Table_2_Field_Multispectral_Data.csv")
# rename period column to PERIOD
current_Dataframe = current_Dataframe.rename(columns={'period': 'PERIOD'})
